# 01 — Create a Lance Dataset

**Purpose:** Foundation notebook for the blueprint. Generates a synthetic multimodal dataset (image + caption + embedding + metadata) with Ray and writes it to a Lance table in Databricks Volumes — the training-ready dataset consumed by `02_cnn_training.ipynb`.

Synthetic data keeps the focus on the Lance workflow itself. There's no external dataset to download, understand, or license — you can run this notebook end to end and get a realistic multimodal table with inline image bytes. (For a controlled Lance-vs-Parquet comparison across scale tiers, see `benchmark/`.)

---

## Why inline binary — not path references?

CNN training requires raw pixels on every batch. Storing a path string instead of image bytes introduces a second I/O hop: Lance read (fast) → object storage GET per image (slow, uncoalesced). At batch size 64 across 100 Ray workers, that's 6,400 concurrent object storage requests per training step. Per-request latency (~10–100ms) compounds and starves the GPU.

Inline binary co-locates pixels with metadata inside Lance fragments. A batch read becomes a single coalesced byte-range read within the fragment file — no per-image request overhead. Lance's blob layout stores heavy image payloads separately from structured metadata columns, so filtering on `category` or a metadata column never touches image bytes.

Lance's fragment-level O(1) addressing makes random batch access cost constant regardless of dataset size — the property that matters most for shuffled training reads.

---

## What this notebook does

1. **Configure paths.** Define the Lance dataset destination at `/Volumes/{catalog}/{schema}/{volume}/frames/` and the dataset size (row count). Fix an RNG seed so the generated data is reproducible.

2. **Define the PyArrow schema.** Declare the table schema upfront so all Ray workers write consistent batches:
   - `id` (int64)
   - `image` (binary — inline JPEG bytes, ~30–300KB)
   - `caption` (string — templated, variable length)
   - `embedding` (list of float32, 512-dim — mimics a CLIP embedding)
   - `category` (string — low-cardinality categorical; the classification label used in `02`)
   - a few numeric metadata columns

3. **Initialize Ray.** Start Ray on the Databricks cluster.

4. **Generate synthetic rows (Ray).** Fan out with `ray.data.range(N).map_batches(generate_fn)`: each batch procedurally draws a `category`, renders a simple image for it, JPEG-encodes the pixels to inline bytes, and fills the caption / embedding / metadata columns. The category-conditioned image is what makes the classification task in `02` learnable.

5. **Write to Lance.** Call `ds.write_lance(frames_path)`. Each Ray write task emits an independent Lance fragment; a single driver-side commit merges the fragment metadata into a new dataset version.

6. **Verify.** Confirm row count and schema. Run `ds.take([0, N//2, N-1])` and time the result — random access should be constant regardless of which rows are requested, demonstrating Lance's O(1) guarantee.

---

**Inputs:** None — data is generated in-notebook.

**Outputs:** Lance `frames` dataset at `/Volumes/{catalog}/{schema}/{volume}/frames/` — inline JPEG bytes, caption, embedding, and metadata, ready for training.

**Next:** `02_cnn_training.ipynb`